# 11 — Final Results and Reproducible Inference
Loads the selected checkpoint, reruns inference on the held-out test period, verifies that the
stored metrics are reproduced exactly, and demonstrates inference for one city.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
from climatesense.utils import read_json
import platform, torch, sklearn
print("Executed:", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), "| Python", platform.python_version(),
      "| torch", torch.__version__, "| sklearn", sklearn.__version__, "| pandas", pd.__version__)
def show(p, width=850):
    display(Image(filename=str(ROOT / p), width=width))

Executed: 2026-10-25 18:08 | Python 3.13.7 | torch 2.14.1+cu126 | sklearn 1.9.1 | pandas 3.1.0


In [2]:
from climatesense.training.trainer import load_trained, predict
from climatesense.evaluation.metrics import compute_metrics
from climatesense.labeling.compound_risk import CLASS_NAMES
st = read_json(ROOT / "results/metrics/selected_model_test.json"); rid = st["selected_run"]
rec = read_json(ROOT / "models/best/model_record.json")   # exported copy of the selected checkpoint (tracked in Git)
assert rec["run_id"] == rid
model, batcher, arr = load_trained(rec)
idx = arr.sample_index(rec["config"]["lookback"], rec["config"]["horizon"], "test")
P, Y, A = predict(model, batcher, idx)
m = compute_metrics(Y, P)
print("checkpoint:", rec["checkpoint"], "| params:", rec["n_params"])
print("recomputed test macro-F1 = %.4f | stored = %.4f" % (m["macro_f1"], st["test_metrics"]["macro_f1"]))
assert abs(m["macro_f1"] - st["test_metrics"]["macro_f1"]) < 1e-6

checkpoint: models/best/mafnet_best.pt | params: 33268
recomputed test macro-F1 = 0.7415 | stored = 0.7415


### Inference example: latest test-period forecasts for one site

In [3]:
p = arr.panel
site = p.loc[p.location_name.eq("New Delhi"), "site_id"].iloc[0] if p.location_name.eq("New Delhi").any() else p.site_id.iloc[0]
code_ = list(arr.site_index).index(site)
mask = arr.site_codes[idx] == code_
out = pd.DataFrame({"forecast_for": p.date.values[idx[mask] + 1], "observed_class": [CLASS_NAMES[int(v)] for v in Y[mask]],
                    "predicted_class": [CLASS_NAMES[int(v)] for v in P[mask].argmax(1)]})
for k in range(4): out[f"P({CLASS_NAMES[k]})"] = P[mask][:, k].round(3)
print(site); out.tail(10)

India|28.6|77.2


,forecast_for,observed_class,predicted_class,P(Low),P(Moderate),P(High),P(Critical)
108,2026-10-03,Critical,Critical,0.0,0.011,0.341,0.648
109,2026-10-04,Critical,Critical,0.0,0.005,0.243,0.752
110,2026-10-05,Critical,Critical,0.0,0.004,0.223,0.773
111,2026-10-06,Critical,Critical,0.0,0.004,0.240,0.756
112,2026-10-07,Critical,Critical,0.0,0.003,0.223,0.773
113,2026-10-08,Critical,Critical,0.0,0.004,0.245,0.751
114,2026-10-09,Critical,Critical,0.0,0.003,0.211,0.787
115,2026-10-10,Critical,Critical,0.0,0.003,0.239,0.757
116,2026-10-11,Critical,Critical,0.0,0.007,0.303,0.689
117,2026-10-12,Critical,Critical,0.0,0.007,0.304,0.689


### Summary of findings

In [4]:
mc = pd.read_csv(ROOT / "results/tables/model_comparison_h1.csv").set_index("model")
ab = pd.read_csv(ROOT / "results/tables/ablation.csv").set_index("variant")
def g(name, col="macro_f1"): return mc.loc[name, col]
PERS = "Persistence (today's class)"
lines = [
 f"* Selected MAF-Net (seed with best validation F1): test macro-F1 {st['test_metrics']['macro_f1']:.3f} (95% CI {st['ci95_macro_f1_site_block_bootstrap'][0]:.3f}–{st['ci95_macro_f1_site_block_bootstrap'][1]:.3f}), accuracy {st['test_metrics']['accuracy']:.3f}, balanced accuracy {st['test_metrics']['balanced_accuracy']:.3f}, macro ROC-AUC {st['test_metrics']['roc_auc_macro_ovr']:.3f}.",
 f"* Five-seed MAF-Net mean test macro-F1 {g('MAF-Net (proposed, attention fusion)'):.3f} ± {g('MAF-Net (proposed, attention fusion)', 'macro_f1_std'):.3f}; concat fusion {g('MAF-Net w/o attention (concat fusion)'):.3f}; tuned GRU {g('GRU (tuned)'):.3f}; gradient boosting {g('Gradient boosting (HistGB)'):.3f}; persistence {g(PERS):.3f}.",
 f"* Ablation (test macro-F1, 3 seeds): climate-only {ab.loc['climate_only','macro_f1']:.3f}, air-only {ab.loc['air_only','macro_f1']:.3f}, dryness-only {ab.loc['dryness_only','macro_f1']:.3f}, all modalities with attention {ab.loc['all_attention','macro_f1']:.3f}.",
]
display(Markdown("\n".join(lines)))

* Selected MAF-Net (seed with best validation F1): test macro-F1 0.741 (95% CI 0.714–0.762), accuracy 0.807, balanced accuracy 0.744, macro ROC-AUC 0.941.
* Five-seed MAF-Net mean test macro-F1 0.741 ± 0.003; concat fusion 0.738; tuned GRU 0.739; gradient boosting 0.734; persistence 0.727.
* Ablation (test macro-F1, 3 seeds): climate-only 0.629, air-only 0.520, dryness-only 0.553, all modalities with attention 0.743.

### Limitations (see `docs/limitations.md`)
Two-and-a-half years of capital-city snapshots; observation-hour bias by longitude; provider
air-quality regime shifts; dryness is a proxy (no SPEI, soil moisture or NDVI); labels are
project-defined and not externally validated; results do not establish human-health impacts.